# 02 — Preprocessing: clean, deduplicate, split **without leakage**

This notebook builds `data/processed/{train,val,test}.parquet`.

The hard part is not the cleaning — it is the **split**. Get that wrong and
every metric downstream is meaningless, which is exactly what happened in the
first version of this project. The story is below.

The real logic lives in [`scripts/build_dataset.py`](../scripts/build_dataset.py)
so it can be re-run and tested; this notebook narrates it step by step.

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent  # notebooks/ -> repo root
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.build_dataset import (
    GROUP_COL,
    KEEP_COLS,
    class_weights,
    clean,
    deduplicate,
    load_raw,
    split,
    verify,
)

PROCESSED_DIR = ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42
print("Writing to:", PROCESSED_DIR)

Writing to: E:\ProjectSummer\VulnerableCheck\data\processed


## 1. The problem: Big-Vul is organised per commit, not per function

Big-Vul mines its rows from **CVE fix commits**. One commit contributes every
function it touched — and those functions come from the same file, the same
author, the same coding style, often with near-identical bodies.

217,007 rows sound like a lot. They come from roughly **4,000 commits**.

In [2]:
raw = load_raw()

per_commit = raw.groupby("commit_id").size()
print(f"rows              : {len(raw):,}")
print(f"distinct commits  : {raw['commit_id'].nunique():,}")
print(f"functions / commit: median {per_commit.median():.0f}, "
      f"mean {per_commit.mean():.1f}, max {per_commit.max():,}")

Loading 'benjis/bigvul' ...


  Hub splits: {'train': 150908, 'validation': 33049, 'test': 33050}
  Combined: 217,007 rows
rows              : 217,007
distinct commits  : 4,019
functions / commit: median 24, mean 54.0, max 1,692


### Why a random split breaks

If you split those **rows** at random, a commit with 24 functions puts some of
them in train and the rest in test. The model then "predicts" test functions it
has effectively already memorised, and the score measures recall of the
training set rather than generalisation.

The first version of this project did exactly that, and scored F1 ≈ 0.95 where
published Big-Vul results sit around 0.3–0.6.

### The official split does not save you

A reasonable instinct is to use the split the dataset ships with. Measure it:

In [3]:
from datasets import load_dataset

dsd = load_dataset("benjis/bigvul")
off_train = set(dsd["train"].to_pandas()["commit_id"])
off_val = set(dsd["validation"].to_pandas()["commit_id"])
off_test = set(dsd["test"].to_pandas()["commit_id"])

print(f"official test commits also in train: "
      f"{len(off_train & off_test):,} / {len(off_test):,} "
      f"({len(off_train & off_test) / len(off_test):.1%})")
print(f"official val  commits also in train: "
      f"{len(off_train & off_val):,} / {len(off_val):,} "
      f"({len(off_train & off_val) / len(off_val):.1%})")

official test commits also in train: 3,208 / 3,215 (99.8%)
official val  commits also in train: 3,249 / 3,256 (99.8%)


**99.8% overlap.** The shipped split is random over rows too, so it carries the
same flaw. That is why this notebook rebuilds the split from scratch instead of
reusing it — a deliberate trade-off: the numbers stop being directly comparable
to papers that use the official split, but they start being *true*.

## 2. Clean: normalise language, drop empties

In [4]:
df = clean(raw)
df.head(3)[["project", "commit_id", "CWE ID", "lang", "vul"]]

Language filter: kept 217,007 / 217,007
  C=213,919, C++=3,088


,project,commit_id,CWE ID,lang,vul
0,libsndfile,708e996c87c5fae77b104ccfeb8f6db784c32074,CWE-119,C,0
1,Chrome,a9cbaa7a40e2b2723cfc2f266c42f4980038a949,CWE-732,C,0
2,xserver,d2f813f7db157fc83abc4b3726821c36ee7e40b1,CWE-189,C,0


## 3. Deduplicate on **normalized** code

Exact-string deduplication misses functions that differ only in indentation or
line breaks. Hashing the whitespace-collapsed source catches those too — and it
turns out they are a quarter of the dataset.

In [5]:
df = deduplicate(df)
print(f"\nRemaining: {len(df):,} functions from {df[GROUP_COL].nunique():,} commits")

vc = df["vul"].value_counts()
print(f"Class balance: not-vul={vc[0]:,}, vul={vc[1]:,}, ratio={vc[0] / vc[1]:.1f}:1")

Deduplication: dropped 53,371 near-duplicates (24.6%), 163,636 remain

Remaining: 163,636 functions from 3,991 commits
Class balance: not-vul=154,933, vul=8,703, ratio=17.8:1


## 4. Split with `StratifiedGroupKFold`, grouped on `commit_id`

Two requirements at once:

- **grouped** — every function from a commit lands in exactly one split, so
  nothing leaks;
- **stratified** — the vulnerable rate stays the same in all three splits, so
  validation and test remain representative.

`StratifiedGroupKFold` does both. Ten folds: one becomes test, one becomes
validation, the remaining eight are train — an 80/10/10 split.

In [6]:
splits = split(df, RANDOM_STATE)

for name, part in splits.items():
    print(f"{name:5s}: {len(part):7,} rows ({len(part) / len(df):5.1%})  "
          f"vul=1: {int(part['vul'].sum()):5,} ({part['vul'].mean():.2%})  "
          f"commits: {part[GROUP_COL].nunique():,}")

train: 130,910 rows (80.0%)  vul=1: 6,962 (5.32%)  commits: 3,208
val  :  16,363 rows (10.0%)  vul=1:   871 (5.32%)  commits: 389
test :  16,363 rows (10.0%)  vul=1:   870 (5.32%)  commits: 394


## 5. Verify — the check the first version was missing

This is the most important cell in the notebook. It **raises** on any overlap,
so a future change that reintroduces leakage fails loudly instead of quietly
producing great-looking numbers.

In [7]:
checks = verify(splits)


Leakage checks (all must be 0):
  OK  commit_overlap_train_test: 0
  OK  commit_overlap_train_val: 0
  OK  commit_overlap_val_test: 0
  OK  code_overlap_train_test: 0
  OK  code_overlap_train_val: 0
  OK  code_overlap_val_test: 0


## 6. Class weights for the imbalance

About 5% of functions are vulnerable (~18:1). Training on that as-is makes the
model lazy — always predicting "not vulnerable" is already ~95% accurate.

Inverse-frequency weights feed `WeightedRandomSampler` in notebook 03, which
draws balanced batches.

In [8]:
weights = class_weights(splits["train"])
print("Class weights:", weights)

with open(PROCESSED_DIR / "class_weights.json", "w", encoding="utf-8") as f:
    json.dump(weights, f, indent=2)

Class weights: {'0': 0.5280843579565624, '1': 9.401752370008618}


In [9]:
# What one sampled epoch actually looks like.
import torch
from torch.utils.data import WeightedRandomSampler

train_df = splits["train"]
sample_weights = train_df["vul"].map(lambda v: weights[str(v)]).to_numpy().copy()
sampler = WeightedRandomSampler(
    weights=torch.as_tensor(sample_weights, dtype=torch.double),
    num_samples=len(sample_weights),
    replacement=True,
)

sampled = train_df["vul"].to_numpy()[list(sampler)]
print("Class balance after weighted sampling:")
print(pd.Series(sampled).value_counts())

Class balance after weighted sampling:
1    65457
0    65453
Name: count, dtype: int64


C:\Users\Admin\AppData\Local\Temp\ipykernel_16744\814357488.py:8: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  weights=torch.as_tensor(sample_weights, dtype=torch.double),


## 7. Save

In [10]:
for name, part in splits.items():
    path = PROCESSED_DIR / f"{name}.parquet"
    part[KEEP_COLS].to_parquet(path, index=False)
    print(f"Saved {path.name}: {part[KEEP_COLS].shape}")

Saved train.parquet: (130910, 7)
Saved val.parquet: (16363, 7)
Saved test.parquet: (16363, 7)


## 8. Sanity check — reload from disk and re-verify independently

Trust the files, not the variables still in memory.

In [11]:
reloaded = {n: pd.read_parquet(PROCESSED_DIR / f"{n}.parquet") for n in ["train", "val", "test"]}

for name, part in reloaded.items():
    print(f"{name:5s} {str(part.shape):>14}  vul={dict(part['vul'].value_counts())}")

tr, va, te = reloaded["train"], reloaded["val"], reloaded["test"]
print("\nRe-checked from disk:")
print("  commit overlap train/test:", len(set(tr[GROUP_COL]) & set(te[GROUP_COL])))
print("  commit overlap train/val :", len(set(tr[GROUP_COL]) & set(va[GROUP_COL])))
assert not (set(tr[GROUP_COL]) & set(te[GROUP_COL]))
assert not (set(tr[GROUP_COL]) & set(va[GROUP_COL]))
print("  OK - no leakage")

train    (130910, 7)  vul={0: np.int64(123948), 1: np.int64(6962)}
val       (16363, 7)  vul={0: np.int64(15492), 1: np.int64(871)}
test      (16363, 7)  vul={0: np.int64(15493), 1: np.int64(870)}

Re-checked from disk:
  commit overlap train/test: 0


  commit overlap train/val : 0
  OK - no leakage


## Summary

`data/processed/` now holds:

| File | Contents |
|------|----------|
| `train.parquet` | ~131K functions, 3,208 commits |
| `val.parquet` | ~16K functions, 389 commits |
| `test.parquet` | ~16K functions, 394 commits |
| `class_weights.json` | inverse-frequency weights for the sampler |
| `split_report.json` | split sizes + the leakage checks, written by the script |

**No commit and no function appears in more than one split.** Metrics measured
on this test set mean what they say.

Next: [`03_finetune.ipynb`](./03_finetune.ipynb).